In [12]:
import os, sys, pickle, torch, numpy as np, dgl
import matplotlib.pyplot as plt
from matplotlib import animation
from matplotlib.colors import ListedColormap

project_path = os.path.abspath(os.path.join(os.getcwd(), '..', ''))
if project_path not in sys.path:
    sys.path.append(project_path)

from hydra.utils import to_absolute_path
from python.create_dgl_dataset import (
    TelemacDataset,
    TelemacDatasetWithQ,
    unpack_dynamic_sample,
    load_liq_hydrograph,
)
from python.CustomMeshGraphNet import MeshGraphNet
from modulus.launch.utils import load_checkpoint

MESH_SLF = "/work/m24046/m24046mrcr/results_data_30min_35_70_maillagex8/Mesh8_corrige.slf"
try:
    from python.python_code.data_manip.extraction.telemac_file import TelemacFile
    from python.create_dgl_dataset import add_mesh_info
    _res_mesh = TelemacFile(MESH_SLF)
    POS, TRIANGLES = add_mesh_info(_res_mesh)
    HAVE_POS = True
except Exception as e:
    print(f"[VISU] Pas de positions (MESH_SLF='{MESH_SLF}'). Raison: {e}")
    POS, TRIANGLES, HAVE_POS = None, None, False

DATA_BASE = "/work/m24046/m24046mrcr/paper/Experience2/Multimesh_8_32.bin"
PCKLS = [
    "/work/m24046/m24046mrcr/results_data_30min_35_70_maillagex8/Group_1_peak_2600_Group_1_peak_2600_0_0-80_interpolated.pkl",
    "/work/m24046/m24046mrcr/results_data_30min_35_70_maillagex8/Group_2_peak_1000_Group_2_peak_1000_0_0-80_interpolated.pkl",
    "/work/m24046/m24046mrcr/results_data_30min_35_70_maillagex8/Group_2_peak_1200_Group_2_peak_1200_0_0-80_interpolated.pkl",
    "/work/m24046/m24046mrcr/results_data_30min_35_70_maillagex8/Group_2_peak_1600_Group_2_peak_1600_0_0-80_interpolated.pkl",
    "/work/m24046/m24046mrcr/results_data_30min_35_70_maillagex8/Group_4_peak_2000_Group_4_peak_2000_0_0-80_interpolated.pkl",
    "/work/m24046/m24046mrcr/results_data_30min_35_70_maillagex8/Group_1_peak_1200_Group_1_peak_1200_0_0-80_interpolated.pkl",
    "/work/m24046/m24046mrcr/results_data_30min_35_70_maillagex8/Group_1_peak_2400_Group_1_peak_2400_0_0-80_interpolated.pkl",
    "/work/m24046/m24046mrcr/results_data_30min_35_70_maillagex8/Group_3_peak_3400_Group_3_peak_3400_0_0-80_interpolated.pkl",
]

USE_Q_FEATURE = False
HYDROS = [
    "/work/m24046/m24046mrcr/dataset_x8_avec_ts/hydrogrammes/generated_hydrographs_Group_1_peak_2600.liq",
    "/work/m24046/m24046mrcr/dataset_x8_avec_ts/hydrogrammes/generated_hydrographs_Group_2_peak_1000.liq",
    "/work/m24046/m24046mrcr/dataset_x8_avec_ts/hydrogrammes/generated_hydrographs_Group_2_peak_1200.liq",
    "/work/m24046/m24046mrcr/dataset_x8_avec_ts/hydrogrammes/generated_hydrographs_Group_2_peak_1600.liq",
    "/work/m24046/m24046mrcr/dataset_x8_avec_ts/hydrogrammes/generated_hydrographs_Group_4_peak_2000.liq",
    "/work/m24046/m24046mrcr/dataset_x8_avec_ts/hydrogrammes/generated_hydrographs_Group_1_peak_1200.liq",
    "/work/m24046/m24046mrcr/dataset_x8_avec_ts/hydrogrammes/generated_hydrographs_Group_1_peak_2400.liq",
    "/work/m24046/m24046mrcr/dataset_x8_avec_ts/hydrogrammes/generated_hydrographs_Group_3_peak_3400.liq",
]
DT_SECONDS = 1800.0

CKPT_DIR = "/work/m24046/m24046mrcr/paper/Experience8/Seed0/"
CKPT_EPOCH = 900

NUM_IN = 10 if USE_Q_FEATURE else 9
NUM_E = 3
NUM_OUT = 3
MP_LAYERS = 10
DO_CONCAT = True
SEGMENTS = 0

DYN_START = 6
DYN_LEN = 4 if USE_Q_FEATURE else 3

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def denorm(xn, mean, std):
    return xn * std + mean

def renorm(x, mean, std):
    return (x - mean) / (std + 1e-12)

def q_from_ts(hydro_tq, ts):
    if ts is None:
        raise ValueError("Le sample ne contient pas ts, impossible d'injecter Q.")
    t_arr, q_arr = hydro_tq
    t_sec = float(ts) * DT_SECONDS
    return float(np.interp(t_sec, t_arr, q_arr, left=q_arr[0], right=q_arr[-1]))

if USE_Q_FEATURE:
    if len(HYDROS) != len(PCKLS):
        raise ValueError(f"HYDROS ({len(HYDROS)}) doit être aligné avec PCKLS ({len(PCKLS)}).")
    ds = TelemacDatasetWithQ(
        name="telemac_test_q",
        data_dir=to_absolute_path(DATA_BASE),
        dynamic_data_files=[to_absolute_path(p) for p in PCKLS],
        hydro_data_files=[to_absolute_path(h) for h in HYDROS],
        split="test",
        ckpt_path=CKPT_DIR,
        normalize=True,
        sequence_length=1,
        overlap=0,
        dt_seconds=DT_SECONDS,
    )
else:
    ds = TelemacDataset(
        name="telemac_test",
        data_dir=to_absolute_path(DATA_BASE),
        dynamic_data_files=[to_absolute_path(p) for p in PCKLS],
        split="test",
        ckpt_path=CKPT_DIR,
        normalize=True,
        sequence_length=1,
        overlap=0,
    )

base_graph = ds.base_graph.to(device)
node_stats = ds.node_stats

g_base = base_graph.to(device)
static_feat = g_base.ndata['static']
DYN_START = static_feat.shape[1]
DYN_LEN = 4 if USE_Q_FEATURE else 3

expected_in = DYN_START + DYN_LEN
if NUM_IN != expected_in:
    raise ValueError(f"NUM_IN={NUM_IN} incompatible avec dataset ({expected_in}).")

mx = np.array([node_stats['h'].item(), node_stats['u'].item(), node_stats['v'].item()], dtype=np.float32)
sx = np.array([node_stats['h_std'].item(), node_stats['u_std'].item(), node_stats['v_std'].item()], dtype=np.float32)
my = np.array([node_stats['delta_h'].item(), node_stats['delta_u'].item(), node_stats['delta_v'].item()], dtype=np.float32)
sy = np.array([node_stats['delta_h_std'].item(), node_stats['delta_u_std'].item(), node_stats['delta_v_std'].item()], dtype=np.float32)

if USE_Q_FEATURE:
    q_mean = float(node_stats['q'].item())
    q_std = float(node_stats['q_std'].item())
else:
    q_mean = 0.0
    q_std = 1.0

model = MeshGraphNet(
    NUM_IN, NUM_E, NUM_OUT,
    processor_size=MP_LAYERS,
    hidden_dim_processor=64,
    hidden_dim_node_encoder=64,
    hidden_dim_edge_encoder=64,
    hidden_dim_node_decoder=64,
    do_concat_trick=DO_CONCAT,
    num_processor_checkpoint_segments=SEGMENTS,
).to(device)
model.eval()

_ = load_checkpoint(
    to_absolute_path(CKPT_DIR),
    models=model,
    device=device,
    epoch=CKPT_EPOCH,
)

def rollout_h_one_file(pkl_path, model, apply_bc=True, hydro_path=None):
    with open(pkl_path, 'rb') as f:
        dynamic_data = pickle.load(f)
    T = len(dynamic_data)
    assert T >= 2

    hydro_tq = None
    if USE_Q_FEATURE:
        if hydro_path is None:
            raise ValueError("hydro_path requis quand USE_Q_FEATURE=True")
        hydro_tq = load_liq_hydrograph(hydro_path)

    g = g_base.clone().to(device)

    x_dyn0, _, ts0 = unpack_dynamic_sample(dynamic_data[0])
    x_dyn0 = x_dyn0.astype(np.float32)
    xn0 = renorm(x_dyn0, mx, sx)

    if USE_Q_FEATURE:
        q0 = q_from_ts(hydro_tq, ts0)
        q0n = (q0 - q_mean) / (q_std + 1e-12) if q_std != 0.0 else (q0 - q_mean)
        q0_col = np.full((xn0.shape[0], 1), q0n, dtype=np.float32)
        xn0_full = np.concatenate([xn0, q0_col], axis=1)
    else:
        xn0_full = xn0

    g.ndata['x'] = torch.cat([static_feat, torch.from_numpy(xn0_full).to(device)], dim=1)

    onehot = g.ndata['x'][:, :4]
    q_mask = (onehot == torch.tensor([0,0,1,0], device=device)).all(dim=1)
    h_mask = (onehot == torch.tensor([0,1,0,0], device=device)).all(dim=1)
    q_mask_np, h_mask_np = q_mask.detach().cpu().numpy(), h_mask.detach().cpu().numpy()

    H_pred = [x_dyn0[:, 0].copy()]
    H_gt = [x_dyn0[:, 0].copy()]

    for t in range(T - 1):
        with torch.no_grad():
            y_pred_n = model(g.ndata['x'], g.edata['x'], g)
        y_pred = denorm(y_pred_n.detach().cpu().numpy(), my, sy)

        xn_t_full = g.ndata['x'][:, DYN_START:DYN_START + DYN_LEN].detach().cpu().numpy()
        xn_t = xn_t_full[:, :3]
        x_t = denorm(xn_t, mx, sx)

        x_t1_pred = x_t + y_pred

        x_t1_gt, _, ts_t1 = unpack_dynamic_sample(dynamic_data[t + 1])
        x_t1_gt = x_t1_gt.astype(np.float32)

        if apply_bc:
            x_t1_pred[q_mask_np, :] = x_t1_gt[q_mask_np, :]
            x_t1_pred[h_mask_np, 0:1] = x_t1_gt[h_mask_np, 0:1]

        H_pred.append(x_t1_pred[:, 0].copy())
        H_gt.append(x_t1_gt[:, 0].copy())

        xn_t1_pred = renorm(x_t1_pred, mx, sx)

        if USE_Q_FEATURE:
            q_t1 = q_from_ts(hydro_tq, ts_t1)
            q_t1n = (q_t1 - q_mean) / (q_std + 1e-12) if q_std != 0.0 else (q_t1 - q_mean)
            q_t1_col = np.full((xn_t1_pred.shape[0], 1), q_t1n, dtype=np.float32)
            xn_t1_full = np.concatenate([xn_t1_pred, q_t1_col], axis=1)
        else:
            xn_t1_full = xn_t1_pred

        g.ndata['x'] = torch.cat([g.ndata['x'][:, :DYN_START], torch.from_numpy(xn_t1_full).to(device)], dim=1)

    H_pred = np.stack(H_pred, axis=0)
    H_gt = np.stack(H_gt, axis=0)
    return H_pred, H_gt

def make_side_by_side_gif(pos, H_pred, H_gt, out_gif="side_by_side_binary.gif", thr=0.01, fps=5):
    T, N = H_pred.shape
    x, y = pos[:, 0], pos[:, 1]
    bin_cmap = ListedColormap(["white", "blue"])

    fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharex=True, sharey=True)
    ax_pred, ax_gt = axes

    hp0 = (H_pred[0] > thr).astype(int)
    hg0 = (H_gt[0] > thr).astype(int)

    sc_pred = ax_pred.scatter(x, y, c=hp0, s=2, vmin=0, vmax=1, cmap=bin_cmap)
    sc_gt = ax_gt.scatter(x, y, c=hg0, s=2, vmin=0, vmax=1, cmap=bin_cmap)

    ax_pred.set_title(f"h prédit (>{thr:.2f} m)")
    ax_gt.set_title(f"h TELEMACH (>{thr:.2f} m)")
    for ax in axes:
        ax.set_aspect("equal")
        ax.set_xlabel("x")
        ax.set_facecolor("white")
    ax_pred.set_ylabel("y")

    def update(frame):
        hp = (H_pred[frame] > thr).astype(int)
        hg = (H_gt[frame] > thr).astype(int)
        sc_pred.set_array(hp)
        sc_gt.set_array(hg)
        fig.suptitle(f"t = {frame}")
        return sc_pred, sc_gt

    ani = animation.FuncAnimation(fig, update, frames=T, interval=1000.0 / fps, blit=False)
    writer = animation.PillowWriter(fps=fps)
    ani.save(out_gif, writer=writer)
    plt.close(fig)
    print(f"GIF sauvegardé dans {out_gif}")

if __name__ == "__main__":
    if not HAVE_POS:
        raise RuntimeError("Impossible de faire la vidéo sans POS (coordonnées nœuds).")

    event_idx = 3
    pkl_path = PCKLS[event_idx]
    hydro_path = HYDROS[event_idx] if USE_Q_FEATURE else None

    base_name = os.path.splitext(os.path.basename(pkl_path))[0]
    print(f"Déroulement sur : {pkl_path}")

    H_pred, H_gt = rollout_h_one_file(pkl_path, model, apply_bc=True, hydro_path=hydro_path)
    out_gif = f"{base_name}_side_by_side.gif"
    make_side_by_side_gif(POS, H_pred, H_gt, out_gif=out_gif, fps=4)


Loading normalization statistics...


[16:24:59 - checkpoint - INFO] Loaded model state dictionary /work/m24046/m24046mrcr/paper/Experience8/Seed0/MeshGraphNet.0.900.mdlus to device cuda
[16:24:59 - checkpoint - INFO] Loaded checkpoint file /work/m24046/m24046mrcr/paper/Experience8/Seed0/checkpoint.0.900.pt to device cuda


Déroulement sur : /work/m24046/m24046mrcr/results_data_30min_35_70_maillagex8/Group_2_peak_1600_Group_2_peak_1600_0_0-80_interpolated.pkl


INFO:matplotlib.animation:Animation.save using <class 'matplotlib.animation.PillowWriter'>


GIF sauvegardé dans Group_2_peak_1600_Group_2_peak_1600_0_0-80_interpolated_side_by_side.gif


In [21]:
# --- Figure: ground truth (top) vs prediction (bottom) at selected horizons ---

def plot_gt_pred_grid(pos, triangles, H_pred, H_gt, out_png="gt_pred_grid.png", thr=0.05, dt_minutes=30, hours=(0,3,6,9,12)):
    import numpy as np
    import matplotlib.pyplot as plt
    import matplotlib.tri as mtri
    from matplotlib.colors import ListedColormap

    T, N = H_pred.shape
    x, y = pos[:, 0], pos[:, 1]

    idx = [int(round(h * 60 / dt_minutes)) for h in hours]
    idx = [min(max(0, i), T-1) for i in idx]

    tri = mtri.Triangulation(x, y, triangles)
    cmap = ListedColormap(["white", "blue"])

    fig, axes = plt.subplots(2, len(hours), figsize=(2.2*len(hours), 4.2), sharex=True, sharey=True)
    if len(hours) == 1:
        axes = np.array(axes).reshape(2, 1)

    for j, (h, t) in enumerate(zip(hours, idx)):
        gt_node = (H_gt[t] > thr).astype(np.int8)
        pr_node = (H_pred[t] > thr).astype(np.int8)

        gt_tri = (gt_node[triangles].mean(axis=1) > 0.5).astype(np.int8)
        pr_tri = (pr_node[triangles].mean(axis=1) > 0.5).astype(np.int8)

        ax_gt = axes[0, j]
        ax_pr = axes[1, j]

        ax_gt.tripcolor(tri, facecolors=gt_tri, vmin=0, vmax=1, cmap=cmap, shading="flat")
        ax_pr.tripcolor(tri, facecolors=pr_tri, vmin=0, vmax=1, cmap=cmap, shading="flat")

        ax_gt.set_title(f"t = {h}h")
        for ax in (ax_gt, ax_pr):
            ax.set_aspect("equal")
            ax.set_xticks([])
            ax.set_yticks([])
            ax.set_facecolor("white")

    axes[0, 0].set_ylabel("Ground truth")
    axes[1, 0].set_ylabel("Prediction")
    fig.suptitle(f"Flood extent (h > {thr:.2f} m)", y=0.98)
    fig.tight_layout(rect=[0, 0, 1, 0.95])
    fig.savefig(out_png, dpi=600, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved: {out_png}")

if HAVE_POS and ("TRIANGLES" in globals()) and (TRIANGLES is not None):
    event_idx = 7
    pkl_path = PCKLS[event_idx]
    hydro_path = HYDROS[event_idx] if USE_Q_FEATURE else None
    base_name = os.path.splitext(os.path.basename(pkl_path))[0]
    H_pred, H_gt = rollout_h_one_file(pkl_path, model, apply_bc=True, hydro_path=hydro_path)
    plot_gt_pred_grid(POS, TRIANGLES, H_pred, H_gt, out_png=f"{base_name}_gt_pred_grid.png", thr=0.01, dt_minutes=30, hours=(0,3,6,9,12))
else:
    print("[VISU] POS/TRIANGLES not available; cannot plot grid.")


Saved: Group_3_peak_3400_Group_3_peak_3400_0_0-80_interpolated_gt_pred_grid.png
